In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Runs in the yolo-train or ir-class conda environment
# (numpy, pandas, scikit-learn, joblib, matplotlib, seaborn).
"""
conda activate yolo-train   # or: conda activate ir-class
"""


In [ ]:
# === Cell 3: Imports ===

import os
import re
import sys
import json
import time
import random
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import f1_score, accuracy_score
import joblib

from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]      = 120
plt.rcParams["savefig.dpi"]     = 150
plt.rcParams["figure.figsize"]  = (8, 5)
plt.rcParams["axes.titlesize"]  = 12
plt.rcParams["axes.labelsize"]  = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10


In [ ]:
# === Cell 4: Config ===

# --- Classifier models to benchmark (name -> .joblib path) ---
# Fitted Pipelines (StandardScaler -> PCA -> clf) saved by the best-preprocessing
# classifier benchmark notebook.
_BENCHMARK_MODELS_DIR = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification"
    "/20260629_Classifier_Benchmark_Best_Preprocessing/results/models"
)
_BENCHMARK_PROJECT = "20260629_Classifier_Benchmark_Best_Preprocessing"

JOBLIB_PATHS = {
    "SVM": _BENCHMARK_MODELS_DIR / f"{_BENCHMARK_PROJECT}_SVM_best_estimator.joblib",
    "kNN": _BENCHMARK_MODELS_DIR / f"{_BENCHMARK_PROJECT}_kNN_best_estimator.joblib",
    "LR":  _BENCHMARK_MODELS_DIR / f"{_BENCHMARK_PROJECT}_LR_best_estimator.joblib",
}

# --- Data sources (same as in the classifier benchmark notebook) ---
SPECTRA_DIR      = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C2_als_z-score")
OBJECT_SPLIT_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/object_split.csv")
GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/spectra_ground_truth_v7.csv")

# Objects not in the segmentation split that go to the test set.
NEW_TEST_OBJECTS = [
    ("HDPE", 13),
    ("PET",  16), ("PET", 21), ("PET", 24), ("PET", 28),
    ("PS",   14), ("PS",  18),
]

# --- Output paths ---
BENCHMARK_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260629_Classifier_Speed_Benchmark")
RESULTS_DIR    = BENCHMARK_ROOT / "results"
for p in [BENCHMARK_ROOT, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME = BENCHMARK_ROOT.name
RANDOM_SEED  = 42

# --- Timing parameters ---
N_WARMUP  = 5     # discarded full passes over the test set
N_REPEATS = 30    # timed full passes (each pass = one predict per test spectrum)

# --- Dataset semantics (same as in the classifier benchmark notebook) ---
FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_(?P<region>body|cap|label)$",
    re.IGNORECASE,
)
OBJECT_ID_WIDTH = 2
VALID_REGIONS   = ["body", "cap", "label"]
VALID_SPLITS    = {"cv", "test"}

REGION_TO_GT_COLUMN = {
    "body":  "body_class",
    "label": "label_class",
    "cap":   "cap_class",
}

# --- Output artefact paths ---
SPEED_BENCHMARK_CSV  = RESULTS_DIR / f"{PROJECT_NAME}_speed_benchmark.csv"
TIMING_RAW_CSV       = RESULTS_DIR / f"{PROJECT_NAME}_timing_raw.csv"
SPEED_COMPARISON_PNG = RESULTS_DIR / f"{PROJECT_NAME}_speed_comparison.png"

print(f"Project:    {PROJECT_NAME}")
print(f"Classifiers to benchmark: {list(JOBLIB_PATHS.keys())}")
print(f"N_WARMUP:   {N_WARMUP}")
print(f"N_REPEATS:  {N_REPEATS}")


In [ ]:
# === Cell 5: Sanity Check ===

import sklearn
print("Python:       ", sys.version.split()[0])
print("numpy:        ", np.__version__)
print("pandas:       ", pd.__version__)
print("scikit-learn: ", sklearn.__version__)
print("Platform:     ", platform.platform())
print()

PATHS_TO_CHECK = [
    ("SPECTRA_DIR",      SPECTRA_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("GROUND_TRUTH_CSV", GROUND_TRUTH_CSV),
    ("BENCHMARK_ROOT",   BENCHMARK_ROOT),
    ("RESULTS_DIR",      RESULTS_DIR),
]
all_ok = True
print("Path checks")
for name, path in PATHS_TO_CHECK:
    kind   = "DIR " if path.is_dir()  else "FILE"
    status = "OK     " if path.exists() else "MISSING"
    print(f"  [{kind}] {status}  {name}: {path}")
    if not path.exists():
        all_ok = False

print("\nModel files")
for clf_name, path in JOBLIB_PATHS.items():
    status = "OK     " if path.exists() else "MISSING"
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f"  [FILE] {status}  {clf_name}: {path.name}  ({size_kb:.1f} KB)")
    if not path.exists():
        all_ok = False


def read_spectrum_file(path):
    """Load a semicolon-delimited wavenumber;reflectance file, return sorted DataFrame."""
    df = pd.read_csv(path, sep=";", header=None,
                     names=["wavenumber", "reflectance"], engine="python")
    if df.shape[1] != 2:
        raise ValueError(f"Expected 2 columns, got {df.shape[1]}: {path}")
    df["wavenumber"]  = pd.to_numeric(df["wavenumber"],  errors="coerce")
    df["reflectance"] = pd.to_numeric(df["reflectance"], errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"Non-numeric values in {path}")
    if df.empty:
        raise ValueError(f"Empty file: {path}")
    return df.sort_values("wavenumber").reset_index(drop=True)


if not all_ok:
    raise RuntimeError("Sanity check FAILED — fix paths before continuing.")
print("\nSanity check PASS")


In [ ]:
# === Cell 6: File Discovery and Parsing ===

# Same parsing as in the classifier benchmark notebook.
# Files are extensionless; filter by FILENAME_REGEX on the stem.

def parse_filename(path):
    """Extract material, object_id, region, object_stem, group_id from a filename."""
    match = FILENAME_REGEX.match(Path(path).stem)
    if not match:
        return None
    d   = match.groupdict()
    mat = d["material"].upper()
    oid = d["object_id"].zfill(OBJECT_ID_WIDTH)
    stem = f"{mat}_obj_{oid}"
    return {
        "path":            str(path),
        "filename":        Path(path).name,
        "prefix_material": mat,
        "object_id":       oid,
        "region":          d["region"].lower(),
        "object_stem":     stem,
        "group_id":        stem,
    }


all_files     = sorted([p for p in SPECTRA_DIR.iterdir() if p.is_file()])
parsed_rows   = []
skipped_files = []
for p in all_files:
    res = parse_filename(p)
    (parsed_rows if res is not None else skipped_files).append(
        res if res is not None else p.name
    )

if skipped_files:
    print(f"WARNING: {len(skipped_files)} files did not match FILENAME_REGEX and are excluded:")
    for name in skipped_files[:10]:
        print(f"  - {name}")

raw_meta_df = pd.DataFrame(parsed_rows)
if raw_meta_df.empty:
    raise RuntimeError(f"No files matched FILENAME_REGEX in {SPECTRA_DIR}.")

print(f"Files discovered:  {len(all_files)}")
print(f"Files parsed:      {len(raw_meta_df)}")
print(f"Unique objects:    {raw_meta_df['group_id'].nunique()}")
print(f"Unique regions:    {sorted(raw_meta_df['region'].unique())}")


In [ ]:
# === Cell 7: Ground Truth and Outer Split ===

# Same split logic as in the classifier benchmark notebook.
# Only the test split is used downstream.

# --- (a) Load ground truth ---
gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";", dtype=str)
gt_df.columns = [c.strip() for c in gt_df.columns]
for c in gt_df.columns:
    gt_df[c] = gt_df[c].str.strip().replace({"": np.nan})

print("Ground-truth columns:", list(gt_df.columns))
print(f"Ground-truth objects: {len(gt_df)}")
gt_indexed = gt_df.set_index("filename")


# --- (b) Assign region-specific true class ---
def lookup_true_class(row):
    """Return the ground-truth polymer class for this spectrum's region, or NaN."""
    stem = row["object_stem"]
    col  = REGION_TO_GT_COLUMN[row["region"]]
    if stem not in gt_indexed.index:
        return np.nan
    return gt_indexed.loc[stem, col]


meta_df = raw_meta_df.copy()
meta_df["true_class"] = meta_df.apply(lookup_true_class, axis=1)
meta_df = meta_df[meta_df["true_class"].notna()].reset_index(drop=True)

# --- (c) Outer split: segmentation split, else NEW_TEST_OBJECTS rule ---
split_df = pd.read_csv(OBJECT_SPLIT_CSV, dtype=str)
split_df.columns = [c.strip() for c in split_df.columns]
split_df["material"]    = split_df["material"].str.upper()
split_df["object_id"]   = split_df["object_id"].str.extract(r"(\d+)")[0].str.zfill(OBJECT_ID_WIDTH)
split_df["object_stem"] = split_df["material"] + "_obj_" + split_df["object_id"]
split_df["split"]       = split_df["split"].str.strip().str.lower()

carryover_split = dict(zip(split_df["object_stem"], split_df["split"]))
NEW_TEST_STEMS  = {
    f"{mat.upper()}_obj_{int(oid):0{OBJECT_ID_WIDTH}d}" for mat, oid in NEW_TEST_OBJECTS
}


def assign_outer_split(object_stem):
    """cv/test for an object: carry over from segmentation split, else new-object rule."""
    if object_stem in carryover_split:
        return carryover_split[object_stem]
    if object_stem in NEW_TEST_STEMS:
        return "test"
    return "cv"


meta_df["split"] = meta_df["object_stem"].map(assign_outer_split)
meta_df = meta_df[meta_df["split"].isin(VALID_SPLITS)].reset_index(drop=True)

ALL_CLASSES = sorted(meta_df["true_class"].unique())
print(f"\nTarget classes ({len(ALL_CLASSES)}): {ALL_CLASSES}")
print(f"Total active spectra: {len(meta_df)}")
print(f"  cv:   {(meta_df['split'] == 'cv').sum()}")
print(f"  test: {(meta_df['split'] == 'test').sum()}")


In [ ]:
# === Cell 8: Test Feature Loading ===

# Load test spectra into X_test / y_test; CV spectra are not needed.

def load_feature_vector(path):
    """Return (reflectance_vector, wavenumber_axis) for one spectrum file."""
    df = read_spectrum_file(path)
    return df["reflectance"].to_numpy(dtype=np.float64), df["wavenumber"].to_numpy()


meta_test = meta_df[meta_df["split"] == "test"].reset_index(drop=True)

print("Loading test spectra ...")
feature_rows, wavenumber_ref, load_errors, n_features = [], None, [], None
for _, row in meta_test.iterrows():
    try:
        vec, wn = load_feature_vector(row["path"])
        if wavenumber_ref is None:
            wavenumber_ref, n_features = wn, len(vec)
            print(f"  Reference axis: {len(wn)} points, wn=[{wn.min():.1f}, {wn.max():.1f}] cm-1")
        if len(vec) != n_features:
            raise ValueError(f"length {len(vec)} != reference {n_features}")
        feature_rows.append(vec)
    except Exception as exc:
        load_errors.append({"filename": row["filename"], "error": str(exc)})

if load_errors:
    print(f"\nWARNING: {len(load_errors)} files failed to load:")
    for e in load_errors:
        print(f"  - {e['filename']}: {e['error']}")
if not feature_rows:
    raise RuntimeError("No test spectra could be loaded.")

X_test = np.vstack(feature_rows)
y_test = meta_test["true_class"].to_numpy()

print(f"\nTest feature matrix: {X_test.shape}  ({X_test.shape[0]} spectra x {X_test.shape[1]} points)")
print(f"Test objects:        {meta_test['group_id'].nunique()}")
print("Test class counts:")
for cls in ALL_CLASSES:
    print(f"  {cls:6s}: {(y_test == cls).sum()}")


In [ ]:
# === Cell 9: Model Loading and Accuracy Check ===

# Load each .joblib pipeline and check that it reaches 100 % accuracy and
# macro-F1 on the test set.

LOADED_MODELS = {}
sanity_rows   = []

print("Loading and verifying classifiers on the test set")
print("=" * 70)

for clf_name, joblib_path in JOBLIB_PATHS.items():
    pipe = joblib.load(joblib_path)
    LOADED_MODELS[clf_name] = pipe

    y_pred   = pipe.predict(X_test)
    acc      = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    size_kb = joblib_path.stat().st_size / 1024

    sanity_rows.append({
        "classifier":      clf_name,
        "test_accuracy":   acc,
        "test_macro_f1":   macro_f1,
        "model_size_kb":   round(size_kb, 1),
        "n_pipeline_steps": len(pipe.named_steps),
        "pca_n_components": pipe.named_steps["pca"].n_components_,
    })

    status = "PASS" if acc == 1.0 and macro_f1 == 1.0 else "MISMATCH"
    print(f"  {clf_name:4s}  accuracy={acc:.4f}  macro-F1={macro_f1:.4f}  "
          f"size={size_kb:.1f} KB  PCA dims={pipe.named_steps['pca'].n_components_}  [{status}]")

sanity_df = pd.DataFrame(sanity_rows)
print("\nSanity check summary:")
display(sanity_df)

# Warn if any classifier is below 100 % accuracy or macro-F1.
_mismatches = sanity_df[
    (sanity_df["test_accuracy"] < 1.0) | (sanity_df["test_macro_f1"] < 1.0)
]
if len(_mismatches):
    print("\nWARNING: the following classifiers did NOT reproduce 100% on the test set:")
    display(_mismatches)
    print("Timing results may be unreliable — check for sklearn version mismatch "
          "or data-loading divergence before proceeding.")
else:
    print("\nAll classifiers reproduced 100% accuracy and macro-F1 on the test set.")


In [ ]:
# === Cell 10: Timing Loop ===

# Per-spectrum latency: each test spectrum is predicted alone, shape (1, n_features),
# as on the sorting line. N_WARMUP discarded passes, then N_REPEATS timed passes
# over all test spectra.

n_test = X_test.shape[0]
print(f"Test spectra:  {n_test}")
print(f"Warm-up:       {N_WARMUP} passes ({N_WARMUP * n_test} calls, discarded)")
print(f"Timed:         {N_REPEATS} passes ({N_REPEATS * n_test} calls per classifier)")
print("=" * 70)

timing_rows = []

for clf_name, pipe in LOADED_MODELS.items():
    print(f"\n{clf_name}: warm-up ...", end="", flush=True)

    # --- Warm-up (discarded) ---
    for _ in range(N_WARMUP):
        for i in range(n_test):
            pipe.predict(X_test[i:i+1])
    print(" done.  Timing ...", end="", flush=True)

    # --- Timed passes ---
    for rep in range(N_REPEATS):
        for i in range(n_test):
            t0 = time.perf_counter()
            pipe.predict(X_test[i:i+1])
            t1 = time.perf_counter()
            timing_rows.append({
                "classifier":  clf_name,
                "repeat":      rep,
                "sample_idx":  i,
                "time_s":      t1 - t0,
            })

    print(" done.")

timing_df = pd.DataFrame(timing_rows)
timing_df["time_ms"] = timing_df["time_s"] * 1000.0

print(f"\nCollected {len(timing_df)} timed predictions.")


In [ ]:
# === Cell 11: Aggregation and Results ===

# Per-classifier summary: mean and std of single-spectrum inference time (ms),
# throughput (spectra / second), and model size on disk.

agg_rows = []
for clf_name in LOADED_MODELS:
    sub = timing_df[timing_df["classifier"] == clf_name]
    mean_ms = sub["time_ms"].mean()
    std_ms  = sub["time_ms"].std(ddof=1)
    throughput = 1000.0 / mean_ms if mean_ms > 0 else np.inf

    # Model size on disk
    size_kb = JOBLIB_PATHS[clf_name].stat().st_size / 1024

    # Test metrics from sanity_df
    san = sanity_df[sanity_df["classifier"] == clf_name].iloc[0]

    agg_rows.append({
        "classifier":        clf_name,
        "mean_ms":           round(mean_ms, 4),
        "std_ms":            round(std_ms, 4),
        "throughput_per_sec": round(throughput, 1),
        "model_size_kb":     round(size_kb, 1),
        "pca_n_components":  int(san["pca_n_components"]),
        "test_accuracy":     san["test_accuracy"],
        "test_macro_f1":     san["test_macro_f1"],
    })

results_df = pd.DataFrame(agg_rows)

print("Speed benchmark results (single-spectrum inference)")
print("=" * 70)
display(results_df)

# Fastest classifier
fastest = results_df.loc[results_df["mean_ms"].idxmin()]
print(f"\nFastest classifier: {fastest['classifier']}  "
      f"({fastest['mean_ms']} +/- {fastest['std_ms']} ms/spectrum, "
      f"{fastest['throughput_per_sec']} spectra/s)")

# Per-repeat aggregation (mean over test spectra within each repeat)
repeat_agg = (
    timing_df.groupby(["classifier", "repeat"])["time_ms"]
    .mean()
    .reset_index()
    .rename(columns={"time_ms": "mean_ms_per_repeat"})
)
print("\nPer-repeat mean inference time (ms/spectrum):")
for clf_name in LOADED_MODELS:
    sub = repeat_agg[repeat_agg["classifier"] == clf_name]["mean_ms_per_repeat"]
    print(f"  {clf_name:4s}: {sub.mean():.4f} +/- {sub.std(ddof=1):.4f} ms  "
          f"(min={sub.min():.4f}, max={sub.max():.4f})")

# Save results
results_df.to_csv(SPEED_BENCHMARK_CSV, index=False)
timing_df.to_csv(TIMING_RAW_CSV, index=False)
print(f"\nSummary saved to:    {SPEED_BENCHMARK_CSV}")
print(f"Raw timings saved to: {TIMING_RAW_CSV}")


In [ ]:
# === Cell 12: Speed Comparison Plot ===
# Standalone: reloads results from SPEED_BENCHMARK_CSV (needs only the Config cell).

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# --- Style config (10 pt at 16 cm LaTeX text width) ---
_FW        = 16 / 2.54
_FONT_SIZE = 10

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

# --- Reload results from disk ---
if not SPEED_BENCHMARK_CSV.exists():
    raise FileNotFoundError(
        f"Speed benchmark CSV not found: {SPEED_BENCHMARK_CSV}\n"
        "Run the benchmark cells first to produce the results CSV."
    )

results_df = pd.read_csv(SPEED_BENCHMARK_CSV)
print(f"Loaded {len(results_df)} rows from {SPEED_BENCHMARK_CSV.name}")

# --- Plot ---
classifiers = results_df["classifier"].tolist()
x_pos       = np.arange(len(classifiers))
_n          = len(classifiers)
_panel_w    = _FW / 2

# Fixed classifier colours (tab10 order SVM, kNN, RF, DT, LR), consistent with
# the classifier benchmark plots.
_CLASSIFIER_COLORS = {
    "SVM": plt.cm.tab10(0),
    "kNN": plt.cm.tab10(1),
    "RF":  plt.cm.tab10(2),
    "DT":  plt.cm.tab10(3),
    "LR":  plt.cm.tab10(4),
}
_colors = [_CLASSIFIER_COLORS[c] for c in classifiers]

def _add_bands(ax, n):
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color="#f0f0f0", zorder=0)

fig, axes = plt.subplots(1, 2, figsize=(_FW, _panel_w))

# --- Left: mean inference time (ms) with std error bars ---
ax = axes[0]
_add_bands(ax, _n)
bars = ax.bar(
    x_pos, results_df["mean_ms"], yerr=results_df["std_ms"],
    capsize=4, color=_colors, edgecolor="black", linewidth=0.5,
    alpha=0.85, zorder=3,
)
ax.set_xticks(x_pos)
ax.set_xticklabels(classifiers)
ax.set_ylabel("Inference Time (ms / spectrum)")
ax.set_title("Single-Spectrum Inference Latency")
ax.set_ylim(0, (results_df["mean_ms"] + results_df["std_ms"]).max() * 1.18)
ax.grid(axis="y", alpha=0.3, zorder=0)
ax.set_box_aspect(1)
for bar, row in zip(bars, results_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + row.std_ms + (results_df["mean_ms"].max() * 0.01),
            f"{row.mean_ms:.3f} ms",
            ha="center", va="bottom", fontsize=_FONT_SIZE - 2, fontweight="bold")

# --- Right: throughput (spectra/s) ---
ax = axes[1]
_add_bands(ax, _n)
bars = ax.bar(
    x_pos, results_df["throughput_per_sec"],
    color=_colors, edgecolor="black", linewidth=0.5,
    alpha=0.85, zorder=3,
)
ax.set_xticks(x_pos)
ax.set_xticklabels(classifiers)
ax.set_ylabel("Throughput (spectra / second)")
ax.set_title("Classification Throughput")
ax.set_ylim(0, results_df["throughput_per_sec"].max() * 1.18)
ax.grid(axis="y", alpha=0.3, zorder=0)
ax.set_box_aspect(1)
for bar, row in zip(bars, results_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + results_df["throughput_per_sec"].max() * 0.01,
            f"{row.throughput_per_sec:.0f}",
            ha="center", va="bottom", fontsize=_FONT_SIZE - 2, fontweight="bold")

fig.suptitle("IR Classifier Speed Benchmark", fontsize=_FONT_SIZE + 1)
fig.tight_layout()
fig.savefig(SPEED_COMPARISON_PNG, bbox_inches="tight", dpi=150)
print(f"Plot saved to: {SPEED_COMPARISON_PNG}")
plt.show()